# 04 — Model training and evaluation

This notebook trains a **baseline** `StandardScaler + LogisticRegression` pipeline and saves it for `scripts/run_demo.py`.

Because the BrainFlow **synthetic** board does not elicit true SSVEP responses, the cells below first build a **toy dataset** (sinusoids + noise) so you can run the full workflow end-to-end. Replace that block with your own labeled windows when you have real data.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np

from acquisition.brainflow_stream import BrainFlowStream
from features.frequency_features import extract_ssvep_feature_vector
from models.classifier import save_sklearn_pipeline, train_logistic_regression
from signal_processing.filters import bandpass_filter
from sklearn.metrics import classification_report
from utils.config import SSVEPConfig

## Build toy trials (two classes)

Class **0 (LEFT)** emphasizes `left_hz`; class **1 (RIGHT)** emphasizes `right_hz`.

In [ ]:
cfg = SSVEPConfig(project_root=ROOT)
fs = BrainFlowStream().sampling_rate()
rng = np.random.default_rng(42)
n_trials = 40
n_samples = int(cfg.window_seconds * fs)
t = np.arange(n_samples) / fs

low, high = cfg.bandpass_band_hz()


def make_trial(label: int) -> np.ndarray:
    if label == 0:
        sig = np.sin(2 * np.pi * cfg.left_hz * t)
    else:
        sig = np.sin(2 * np.pi * cfg.right_hz * t)
    eeg = np.stack([sig, sig, sig], axis=0)
    eeg += 0.4 * rng.standard_normal(eeg.shape)
    return bandpass_filter(eeg, fs, low, high)


X_list = []
y_list = []
for i in range(n_trials):
    lab = i % 2
    eeg = make_trial(lab)
    feat = extract_ssvep_feature_vector(eeg, fs, (cfg.left_hz, cfg.right_hz))
    X_list.append(feat)
    y_list.append(lab)

X = np.stack(X_list, axis=0)
y = np.array(y_list, dtype=np.int64)
X.shape, y.shape

## Train / evaluate / save

In [ ]:
model, metrics = train_logistic_regression(X, y)
metrics

In [ ]:
y_hat = model.predict(X)
print(classification_report(y, y_hat, target_names=["LEFT", "RIGHT"]))

In [ ]:
cfg.models_dir.mkdir(parents=True, exist_ok=True)
out_path = cfg.models_dir / "ssvep_pipeline.joblib"
save_sklearn_pipeline(model, out_path)
print("Saved:", out_path)

## Optional: train on real labeled windows

If you saved `X` with shape `(n_trials, n_channels, n_samples)` and matching `y` in `{0,1}` from a calibration session, vectorize with the same `extract_ssvep_feature_vector` call used above for each trial.